In [1]:
import pandas as pd
import numpy as np

TARGETS = ["theta"]

SETS =  [
    "ZZxReto", # Train
    "ZZy1", # Train
    "ZZx2",  # Val
    "ZZy2", # Val
    "LSG-1", # Test
    "LSG-2", # Test
    "ZZx1-inv", # Test
    "ZZx1",  # Test
    "ZZx2-inv", # Test
    "semiCirc", # Test
]

In [2]:
results_1l = pd.read_excel("resultados-1l.xlsx")
# results_2l = pd.read_excel("resultados-2l.xlsx")
# results_3l = pd.read_excel("resultados-3l.xlsx")

# results = pd.concat(
    # [results_1l, results_2l, results_3l],
    # ignore_index=True
# )
results = results_1l

import re

def fix_column_name(col):
    # Ex: "MSE_ZZxReto_theta" -> "R2_ZZxReto_dtheta"
    match = re.match(r"^MSE_(.+)_([^_]+)$", col)
    if match:
        title, name = match.groups()
        return f"R2_{title}_d{name}"
    return col

results.columns = [fix_column_name(c) for c in results.columns]

In [3]:
results

,model,Neurons,Ld,Lp,reg,seed,R2_ZZxReto_theta,R2_ZZxReto_dtheta,R2_ZZy1_theta,R2_ZZy1_dtheta,...,R2_LSG_2_theta,R2_LSG_2_dtheta,R2_ZZx1_inv_theta,R2_ZZx1_inv_dtheta,R2_ZZx1_theta,R2_ZZx1_dtheta,R2_ZZx2_inv_theta,R2_ZZx2_inv_dtheta,R2_semiCirc_theta,R2_semiCirc_dtheta
0,model_arch1_r0.01_Ld0.3_Lp0.7_seed9644,[1],0.3,0.7,0.01,9644,0.733057,0.476714,-3.011702,0.270240,...,-1.695874,0.271321,0.566030,0.436876,0.321922,0.377274,-0.441283,0.082723,-8.036751,0.083170
1,model_arch1_r0.01_Ld0.3_Lp0.7_seed2553,[1],0.3,0.7,0.01,2553,0.739420,0.482715,-3.065457,0.273468,...,-1.645773,0.277344,0.585662,0.442353,0.335997,0.378662,-0.496132,0.075913,-8.614595,0.084406
2,model_arch1_r0.01_Ld0.3_Lp0.7_seed6913,[1],0.3,0.7,0.01,6913,0.461696,0.272392,-1.266978,0.199365,...,-1.578698,0.070287,-0.245256,0.172206,0.393823,0.305170,0.160164,0.153581,-3.263292,-0.103057
3,model_arch1_r0.01_Ld0.3_Lp0.7_seed4834,[1],0.3,0.7,0.01,4834,0.499445,0.276658,-1.500023,0.198762,...,-1.749962,0.066322,-0.241009,0.172061,0.356355,0.313394,0.143015,0.151597,-3.539193,-0.114465
4,model_arch1_r0.01_Ld0.3_Lp0.7_seed8981,[1],0.3,0.7,0.01,8981,0.736287,0.481590,-3.325805,0.268541,...,-1.981938,0.269117,0.526524,0.437206,0.293766,0.385982,-0.456774,0.078276,-8.224858,0.072398
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022,model_arch100_r0.9_Ld0.7_Lp0.3_seed4733,[100],0.7,0.3,0.90,4733,0.617318,0.496319,-7.057327,0.255248,...,0.366375,0.402151,0.805186,0.593170,0.271080,0.412880,-0.151156,0.359809,-11.011305,0.190002
2023,model_arch100_r0.9_Ld0.7_Lp0.3_seed1242,[100],0.7,0.3,0.90,1242,0.444697,0.470078,-7.769007,0.235922,...,0.413512,0.404564,0.665665,0.579663,0.273766,0.420692,-0.851581,0.160333,-9.545569,0.216776
2024,model_arch100_r0.9_Ld0.7_Lp0.3_seed2799,[100],0.7,0.3,0.90,2799,0.457218,0.504426,-9.380935,0.246788,...,0.094971,0.400819,0.668138,0.579100,0.420085,0.474173,-0.813497,0.122135,-11.297277,0.175529
2025,model_arch100_r0.9_Ld0.7_Lp0.3_seed1858,[100],0.7,0.3,0.90,1858,-0.126809,0.483458,-9.965407,0.226078,...,0.260911,0.405507,-0.241250,0.575144,0.067590,0.430448,-0.969080,0.224469,-8.475919,0.248801


In [4]:
# 🔹 categorização dos sets (baseada nos comentários originais)
SETS_CATEGORY = {
    "ZZxReto":  "Train",
    "ZZy1":     "Train",
    "ZZx2":     "Val",
    "ZZy2":     "Val",
    "LSG-1":    "Test",
    "LSG-2":    "Test",
    "ZZx1-inv": "Test",
    "ZZx1":     "Test",
    "ZZx2-inv": "Test",
    "semiCirc": "Test",
}

def col_name(s, target):
    # sanitiza "-" pra "_" pra bater com o nome real da coluna, se for o caso
    return f"R2_{s.replace('-', '_')}_{target}"

best_models_tables = {}
N = 10  # top modelos

w_val = 0.33
w_train = 0.33
w_test = 0.33

for target in TARGETS:

    # 🔹 sets de Train, Val e Test
    train_sets = [s for s, cat in SETS_CATEGORY.items() if cat == "Train"]
    val_sets   = [s for s, cat in SETS_CATEGORY.items() if cat == "Val"]
    test_sets  = [s for s, cat in SETS_CATEGORY.items() if cat == "Test"]

    train_cols = [col_name(s, target) for s in train_sets]
    val_cols   = [col_name(s, target) for s in val_sets]
    test_cols  = [col_name(s, target) for s in test_sets]

    # 🔹 garantir que só usamos colunas existentes
    train_cols = [c for c in train_cols if c in results.columns]
    val_cols   = [c for c in val_cols if c in results.columns]
    test_cols  = [c for c in test_cols if c in results.columns]

    r2_all_cols = train_cols + val_cols + test_cols

    if not r2_all_cols:
        print(f"⚠️ Nenhuma coluna Train/Val/Test encontrada para target={target}, pulando.")
        continue

    df = results.copy()

    # 🔹 remover linhas onde QUALQUER R2 (Train/Val/Test) < 0
    # df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    df["R2_train_mean"] = df[train_cols].mean(axis=1) if train_cols else np.nan
    df["R2_val_mean"]   = df[val_cols].mean(axis=1) if val_cols else np.nan
    df["R2_test_mean"]  = df[test_cols].mean(axis=1) if test_cols else np.nan

    # =========================
    # 🔹 SCORE
    # =========================
    df["R2_std"] = df[r2_all_cols].std(axis=1)

    df["Score"] = (
        w_train * df["R2_train_mean"] +
        w_val   * df["R2_val_mean"] +
        w_test  * df["R2_test_mean"]
        - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    df_sorted = df.sort_values(by="Score", ascending=False)
    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    final_cols = ["model", "Neurons"] + r2_all_cols + [
        "R2_train_mean", "R2_val_mean", "R2_test_mean", "Score"
    ]
    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 10 MODELOS - theta


,model,Neurons,R2_train_mean,R2_val_mean,R2_test_mean,Score
1127,model_arch56_r0.01_Ld0.3_Lp0.7_seed4733,[56],0.089192,-0.619113,-1.605364,-0.913123
1215,model_arch60_r0.9_Ld0.3_Lp0.7_seed1858,[60],-0.205254,-0.691873,-1.294077,-0.949319
943,model_arch47_r0.9_Ld0.7_Lp0.3_seed9599,[47],0.335465,-1.872460,-0.887175,-0.960729
606,model_arch30_r0.9_Ld0.7_Lp0.3_seed4976,[30],-0.026783,-1.679027,-0.829950,-0.985593
941,model_arch47_r0.01_Ld0.7_Lp0.3_seed4976,[47],-0.024555,-1.463310,-1.024739,-1.004769
994,model_arch50_r0.9_Ld0.3_Lp0.7_seed8263,[50],-0.128642,-1.339576,-1.159066,-1.072650
1889,model_arch94_r0.01_Ld0.3_Lp0.7_seed2799,[94],-0.773387,-0.633744,-1.151030,-1.093023
1284,model_arch63_r0.9_Ld0.7_Lp0.3_seed2799,[63],-1.062742,-1.111338,-0.661124,-1.103762
1164,model_arch57_r0.9_Ld0.7_Lp0.3_seed2799,[57],0.174970,-1.970122,-1.042245,-1.109880
1266,model_arch62_r0.9_Ld0.7_Lp0.3_seed5166,[62],-0.775336,-0.998822,-1.038111,-1.136916



📊 MÉTRICAS COMPLETAS - TOP 10 (theta)


,model,Neurons,R2_ZZxReto_theta,R2_ZZy1_theta,R2_ZZx2_theta,R2_ZZy2_theta,R2_LSG_1_theta,R2_LSG_2_theta,R2_ZZx1_inv_theta,R2_ZZx1_theta,R2_ZZx2_inv_theta,R2_semiCirc_theta,R2_train_mean,R2_val_mean,R2_test_mean,Score
1127,model_arch56_r0.01_Ld0.3_Lp0.7_seed4733,[56],0.191373,-0.012989,-1.182116,-0.056110,-0.169710,-2.347187,0.108968,-0.954036,0.255590,-6.525811,0.089192,-0.619113,-1.605364,-0.913123
1215,model_arch60_r0.9_Ld0.3_Lp0.7_seed1858,[60],0.587018,-0.997525,-0.696369,-0.687378,-0.090986,-1.174162,0.631931,-0.219022,0.230001,-7.142225,-0.205254,-0.691873,-1.294077,-0.949319
943,model_arch47_r0.9_Ld0.7_Lp0.3_seed9599,[47],0.594156,0.076774,-0.390489,-3.354432,-1.457035,-0.699162,0.504426,0.201614,0.080761,-3.953654,0.335465,-1.872460,-0.887175,-0.960729
606,model_arch30_r0.9_Ld0.7_Lp0.3_seed4976,[30],0.569031,-0.622598,-0.454388,-2.903665,-1.257628,-0.644410,0.597448,0.241081,-0.016814,-3.899379,-0.026783,-1.679027,-0.829950,-0.985593
941,model_arch47_r0.01_Ld0.7_Lp0.3_seed4976,[47],0.639334,-0.688443,-0.669711,-2.256909,-0.653121,-0.939903,0.594381,0.117441,0.022161,-5.289394,-0.024555,-1.463310,-1.024739,-1.004769
994,model_arch50_r0.9_Ld0.3_Lp0.7_seed8263,[50],0.658498,-0.915781,-0.226372,-2.452780,-0.465041,-0.936514,0.610093,0.085372,-0.000778,-6.247531,-0.128642,-1.339576,-1.159066,-1.072650
1889,model_arch94_r0.01_Ld0.3_Lp0.7_seed2799,[94],0.684242,-2.231017,0.054839,-1.322327,-0.203171,0.011149,0.721047,0.220424,-0.082012,-7.573615,-0.773387,-0.633744,-1.151030,-1.093023
1284,model_arch63_r0.9_Ld0.7_Lp0.3_seed2799,[63],0.514949,-2.640432,-0.708753,-1.513923,-0.840424,0.258625,0.727850,0.234771,0.226126,-4.573690,-1.062742,-1.111338,-0.661124,-1.103762
1164,model_arch57_r0.9_Ld0.7_Lp0.3_seed2799,[57],0.608537,-0.258597,-0.824997,-3.115247,-0.870647,-1.133594,0.473053,0.072923,0.072468,-4.867675,0.174970,-1.970122,-1.042245,-1.109880
1266,model_arch62_r0.9_Ld0.7_Lp0.3_seed5166,[62],0.652891,-2.203564,-0.900299,-1.097344,-0.296182,-0.598773,0.743886,0.019166,0.263710,-6.360473,-0.775336,-0.998822,-1.038111,-1.136916


In [5]:
final_table.to_excel("BestModels-1l.xlsx")